In [ ]:
##################Phase 2-Urban Flood Risk: Lagos & Durban ##########################
#######STEP 7- Scenario modelling and cost-benefit analysis

##Purpose
#This notebook evaluates flood mitigation scenarios by estimating reductions in flood 
#damages under different intervention strategies and comparing implementation costs with projected economic benefits.

## ResearchContext
#Scenario modelling supports evidence-based decision-making by exploring how 
#alternative adaptation strategies may reduce future flood risk and associated economic losses.

##Input Data
#Regression model outputs
#Spatial flood risk maps
#Historical flood damage estimates
#Mitigation cost assumptions

##Methods
#Scenario development
#Damage reduction modelling
#Cost-benefit analysis
#Benefit-cost ratio calculation
#Comparative evaluation

##Outputs
#Scenario comparison tables
#Cost-benefit ratios
#Estimated economic savings


###STEP 7a- scenario modelling setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

plt.style.use("ggplot")

In [ ]:
#Step 5 models
import joblib

models = {
    "Lagos": joblib.load("lagos_damage_model.pkl"),
    "Durban": joblib.load("durban_damage_model.pkl")
}

datasets = {
    "Lagos": joblib.load("lagos_model.pkl"),
    "Durban": joblib.load("durban_model.pkl")
}

In [ ]:
print(models)
print(datasets)

In [ ]:
#STEP 6 spatial risk results

risk_summary = pd.DataFrame({

    "City": [

        "Lagos",
        "Durban"

    ],

    "Mean_Risk": [

        0.454,
        0.428

    ],

    "High_Risk_Percent": [

        18.9,
        8.5

    ]

})

print(risk_summary)

In [ ]:
#Baseline damage
#Predict expected annual damage using the most recent observations from each city's regression dataset.

for city in ["Lagos", "Durban"]:

    df = datasets[city]
    model = models[city]

    latest = df.iloc[[-1]].copy().reset_index(drop=True)

    latest["log_population"] = np.log(latest["Urban_population"])

    X = latest[
        [
            "Precipitation_mm",
            "Temperature_C",
            "log_population"
        ]
    ]

    predicted_log_damage = model.predict(X).item()

    predicted_damage = np.exp(predicted_log_damage) - 1

    observed_damage = latest["damage"].values[0]

    baseline_damage[city] = {
        "Predicted": float(predicted_damage),
        "Observed": float(observed_damage)
    }


In [ ]:
print(baseline_damage)
baseline_df = pd.DataFrame(baseline_damage).T

print("\nBaseline annual damage estimates")
print(baseline_df.round(2))

In [ ]:
#Future assumptions
#These are scenario assumptions.
#They are NOT observations.
#They are based on expected urban growth and increasing climate pressure.

projection_years = np.arange(

    2025,
    2051

)

growth_rates = {

    "Lagos":0.050,

    "Durban":0.030

}

discount_rate = 0.05

In [ ]:
#Rather than one scenario, let's include several.

#Mitigation options 
#Effectiveness values are taken from the literature.
#These represent approximate reductions in expected annual flood damage.


interventions = {

    "Lagos":[

        {

            "name":"Drainage Upgrade",

            "cost":500,

            "annual_cost":20,

            "effectiveness":0.40

        },

        {

            "name":"Early Warning",

            "cost":80,

            "annual_cost":8,

            "effectiveness":0.18

        },

        {

            "name":"Combined Strategy",

            "cost":560,

            "annual_cost":28,

            "effectiveness":0.52

        }

    ],

    "Durban":[

        {

            "name":"Catchment Restoration",

            "cost":220,

            "annual_cost":12,

            "effectiveness":0.35

        },

        {

            "name":"Early Warning",

            "cost":55,

            "annual_cost":5,

            "effectiveness":0.16

        },

        {

            "name":"Combined Strategy",

            "cost":270,

            "annual_cost":17,

            "effectiveness":0.46

        }

    ]

}

In [ ]:
#Aajust intervention effectiveness using results from step 6
#Instead of treating every city the same, the intervention effectiveness is adjusted by the current spatial flood risk.
#Cities with larger high-risk areas require more effort to achieve the same reduction in flood losses.


risk_factor = {}

for _, row in risk_summary.iterrows():

    city = row["City"]

    risk_factor[city] = row["Mean_Risk"]

In [ ]:
#Helper function 1
#Project future damage

def project_damage(

    baseline,

    growth,

    years

):

    t = years - years[0]

    return baseline * (1 + growth) ** t

In [ ]:
#Helper function 2
#Apply mitigation 

def apply_intervention(

    damages,

    effectiveness,

    city

):

    adjusted_effectiveness = (

        effectiveness *

        (1 - 0.30 * risk_factor[city])

    )

    reduced_damage = damages * (

        1 -

        adjusted_effectiveness

    )

    avoided_damage = damages - reduced_damage

    return reduced_damage, avoided_damage

In [ ]:
#Helper function 3
#Discount future values.
#Net present value

def calculate_npv(

    cashflows,

    rate

):

    years = np.arange(

        len(cashflows)

    )

    return np.sum(

        cashflows /

        (1 + rate) ** years

    )

In [ ]:
#Check setup
print("STEP 7a READY")

print("\nBaseline damages")
print(baseline_df.round(2))

print("\nProjection years")
print(
    projection_years[0],
    "to",
    projection_years[-1]
)

print("\nDiscount rate")
print(discount_rate)

print("\nRisk factors")
print(risk_factor)

print("\nAvailable interventions")

for city in interventions:

    print(f"\n{city}")

    for i in interventions[city]:

        print(
            "-",
            i["name"]
        )

In [ ]:
######STEP 7b- Scenario projections (2025–2050)
import numpy as np
import pandas as pd

##Time horizon
YEARS = np.arange(2025, 2051)
T = len(YEARS)
t = np.arange(T)


#Discount rate (for later NPV)
DISCOUNT_RATE = 0.05


##1. Baseline damage from step 5 regression outputs S
#We anchor to Step 6 + Step 5 results (empirical + spatial risk)

BASELINE_DAMAGE = {
    "Lagos":  180,   #approx EM-DAT mean used in Step 7a setup
    "Durban": 45
}

#Growth assumption derived from Step 6 risk + urban expansion signal
#(higher risk concentration in Lagos -> faster escalation)
GROWTH_RATE = {
    "Lagos": 0.055,
    "Durban": 0.035
}



##2. STEP 6 Spatial risk -> Mitigation effectiveness 
#We use Step 6 outputs:
#Lagos: higher high-risk % (18.9%)
#Durban: lower high-risk % (8.5%)

#This converts spatial risk pressure into intervention response capacity

RISK_PRESSURE = {
    "Lagos": 0.60,   #higher exposure -> interventions less efficient initially
    "Durban": 0.75   # ore stable terrain -> interventions more effective
}


##3. Interventions (mitigation scenarios)

INTERVENTIONS = {
    "Lagos": {
        "Drainage Upgrade": {
            "effectiveness": 0.45,
            "cost_upfront": 500,
            "cost_annual": 20,
            "lag": 3
        },
        "Early Warning": {
            "effectiveness": 0.20,
            "cost_upfront": 80,
            "cost_annual": 8,
            "lag": 1
        }
    },
    "Durban": {
        "Retention + Vegetation": {
            "effectiveness": 0.40,
            "cost_upfront": 200,
            "cost_annual": 12,
            "lag": 4
        },
        "Early Warning": {
            "effectiveness": 0.18,
            "cost_upfront": 50,
            "cost_annual": 5,
            "lag": 1
        }
    }
}



##4. Damage projection function (no intervention) 

def project_no_intervention(city):
    base = BASELINE_DAMAGE[city]
    g = GROWTH_RATE[city]

    damage = base * (1 + g) ** t

    return pd.DataFrame({
        "year": YEARS,
        "damage_no_intervention": damage
    })


##5. Damage projection function (with intervention)

def project_with_intervention(city, intervention_name):

    base = BASELINE_DAMAGE[city]
    g = GROWTH_RATE[city]

    inter = INTERVENTIONS[city][intervention_name]

    lag = inter["lag"]
    eff = inter["effectiveness"]

    #baseline trajectory
    damage_base = base * (1 + g) ** t

    #ramp-up effect (based on Step 6 risk realism)
    ramp = np.minimum(t / lag, 1)

    #adjust effectiveness by spatial risk pressure (Step 6 input)
    adjusted_eff = eff * RISK_PRESSURE[city]

    reduction = ramp * adjusted_eff

    damage_with = damage_base * (1 - reduction)

    return pd.DataFrame({
        "year": YEARS,
        "damage_no_intervention": damage_base,
        "damage_with_intervention": damage_with,
        "avoided_damage": damage_base - damage_with
    })



##6. Test run

for city in ["Lagos", "Durban"]:
    print("\n", city)

    df = project_no_intervention(city)
    print("No intervention end value:", df["damage_no_intervention"].iloc[-1])

    for k in INTERVENTIONS[city]:
        df2 = project_with_intervention(city, k)
        print(k, "end value:", df2["damage_with_intervention"].iloc[-1])

In [ ]:
##########STEP 7c-cost-benefit analysis (NPV, BCR and Break-even)
#This step answers: Is it economically worthwhile to invest in flood mitigation compared with doing nothing?

summary_rows = {}

for city in ["Lagos", "Durban"]:

    summary_rows[city] = {}

    print("\n")
    print("="*65)
    print(city.upper())
    print("="*65)

    for intervention in INTERVENTIONS[city]:

        df = project_with_intervention(
            city,
            intervention
        ).copy()

        info = INTERVENTIONS[city][intervention]

        
        ####Intervention costs
        df["cost"] = info["cost_annual"]

        #add capital investment in first year
        df.loc[df.index[0], "cost"] += info["cost_upfront"]

        ####Net benefit
        df["net_benefit"] = (
            df["avoided_damage"]
            - df["cost"]
        )

        ####Discount factor
        df["discount_factor"] = (
            1 /
            (1 + DISCOUNT_RATE)
            ** np.arange(len(df))
        )

        ####Present Value
        df["discounted_net"] = (
            df["net_benefit"]
            * df["discount_factor"]
        )

        df["cumulative_npv"] = (
            df["discounted_net"]
            .cumsum()
        )

        ####Totals
        total_cost = df["cost"].sum()

        total_avoided = df["avoided_damage"].sum()

        final_npv = df["cumulative_npv"].iloc[-1]

        benefit_cost_ratio = (
            total_avoided /
            total_cost
        )

        ####Break-even year
        positive = df[df["cumulative_npv"] > 0]

        if len(positive) > 0:
            break_even = int(
                positive.iloc[0]["year"]
            )
        else:
            break_even = "Not reached"

        ####Store
        summary_rows[city][intervention] = df

        ####Print
        print(f"\nIntervention : {intervention}")

        print(f"Total Cost (USD M): {total_cost:,.1f}")

        print(f"Damage Avoided (USD M): {total_avoided:,.1f}")

        print(f"Final NPV (USD M): {final_npv:,.1f}")

        print(f"Benefit-Cost Ratio: {benefit_cost_ratio:.2f}")

        print(f"Break-even Year: {break_even}")

In [ ]:
########STEP 7c- summary table 

rows = []

for city in summary_rows:

    for intervention in summary_rows[city]:

        df = summary_rows[city][intervention]

        total_cost = df["cost"].sum()

        total_avoided = df["avoided_damage"].sum()

        final_npv = df["cumulative_npv"].iloc[-1]

        bcr = total_avoided / total_cost

        positive = df[df["cumulative_npv"] > 0]

        if len(positive) > 0:
            break_even = int(
                positive.iloc[0]["year"]
            )
        else:
            break_even = np.nan

        rows.append({

            "City": city,

            "Intervention": intervention,

            "Total Cost (USD M)": round(total_cost,1),

            "Avoided Damage (USD M)": round(total_avoided,1),

            "NPV (USD M)": round(final_npv,1),

            "Benefit-Cost Ratio": round(bcr,2),

            "Break-even Year": break_even

        })

summary_df = pd.DataFrame(rows)

print(summary_df)

summary_df

In [ ]:
#######STEP 7d.1-Damage trajectories 
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16,6)
)

for ax, city in zip(
    axes,
    ["Lagos","Durban"]
):

    #No intervention
    base = project_no_intervention(city)

    ax.plot(
        base["year"],
        base["damage_no_intervention"],
        linewidth=3,
        linestyle="--",
        label="No intervention"
    )

    #Each intervention
    for intervention in INTERVENTIONS[city]:

        df = summary_rows[city][intervention]

        ax.plot(
            df["year"],
            df["damage_with_intervention"],
            linewidth=2,
            label=intervention
        )

    ax.set_title(city)

    ax.set_xlabel("Year")

    ax.set_ylabel(
        "Annual Flood Damage (USD Million)"
    )

    ax.grid(alpha=0.3)

    ax.legend()

plt.suptitle(
    "Flood Damage Projection (2025–2050)",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "Step7_Damage_Trajectories.png",
    dpi=300
)

plt.show()

In [ ]:
##STEP 7d.2- Cumulative NPV

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16,6)
)

for ax, city in zip(
    axes,
    ["Lagos","Durban"]
):

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1,
        color="black"
    )

    for intervention in INTERVENTIONS[city]:

        df = summary_rows[city][intervention]

        ax.plot(
            df["year"],
            df["cumulative_npv"],
            linewidth=2,
            label=intervention
        )

    ax.set_title(city)

    ax.set_xlabel("Year")

    ax.set_ylabel(
        "Cumulative NPV (USD Million)"
    )

    ax.grid(alpha=0.3)

    ax.legend()

plt.suptitle(
    "Net Present Value of Flood Mitigation Investments",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "Step7_Cumulative_NPV.png",
    dpi=300
)

plt.show()

In [ ]:
##STEP 7d.3-Benefit-cost ratio

fig, ax = plt.subplots(
    figsize=(10,6)
)

plot_df = summary_df.copy()

labels = (
    plot_df["City"]
    + "\n"
    + plot_df["Intervention"]
)

bars = ax.bar(
    labels,
    plot_df["Benefit-Cost Ratio"]
)

for bar in bars:

    value = bar.get_height()

    ax.text(
        bar.get_x()+bar.get_width()/2,
        value+0.05,
        f"{value:.2f}",
        ha="center"
    )

ax.axhline(
    1,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Break-even (BCR = 1)"
)

ax.set_ylabel(
    "Benefit-Cost Ratio"
)

ax.set_title(
    "Economic Performance of Flood Mitigation Strategies"
)

ax.legend()

plt.tight_layout()

plt.savefig(
    "Step7_BCR.png",
    dpi=300
)

plt.show()

In [ ]:
##STEP 7d.4-Summary Dushboard

fig = plt.figure(
    figsize=(18,10)
)

gs = fig.add_gridspec(
    2,
    2
)


##Damage trajectories


ax1 = fig.add_subplot(gs[0,0])

base = project_no_intervention("Lagos")

ax1.plot(
    base["year"],
    base["damage_no_intervention"],
    linewidth=3,
    linestyle="--",
    label="No intervention"
)

for intervention in INTERVENTIONS["Lagos"]:

    df = summary_rows["Lagos"][intervention]

    ax1.plot(
        df["year"],
        df["damage_with_intervention"],
        linewidth=2,
        label=intervention
    )

ax1.set_title("Lagos")
ax1.legend()

############################

ax2 = fig.add_subplot(gs[0,1])

base = project_no_intervention("Durban")

ax2.plot(
    base["year"],
    base["damage_no_intervention"],
    linewidth=3,
    linestyle="--",
    label="No intervention"
)

for intervention in INTERVENTIONS["Durban"]:

    df = summary_rows["Durban"][intervention]

    ax2.plot(
        df["year"],
        df["damage_with_intervention"],
        linewidth=2,
        label=intervention
    )

ax2.set_title("Durban")
ax2.legend()

###############################

ax3 = fig.add_subplot(gs[1,:])

bars = ax3.bar(
    labels,
    plot_df["Benefit-Cost Ratio"]
)

ax3.axhline(
    1,
    color="red",
    linestyle="--"
)

for bar in bars:

    ax3.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+0.05,
        f"{bar.get_height():.2f}",
        ha="center"
    )

ax3.set_ylabel("Benefit-Cost Ratio")

ax3.set_title(
    "Comparison of Flood Mitigation Strategies"
)

plt.suptitle(
    "Scenario Modelling (2025–2050): Flood Risk and Economic Evaluation",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "Step7_Dashboard.png",
    dpi=300
)

plt.show()

In [ ]:
#Summary

##Main Outcomes
#Evaluated alternative flood mitigation strategies.
#Estimated the economic benefits associated with each intervention.
#Compared adaptation options using cost-benefit metrics.
#Synthesised statistical, spatial and economic findings to support flood risk management.

##Limitations
#Scenario outcomes depend on assumed intervention costs and effectiveness.
#Results should be interpreted as indicative planning estimates rather than exact forecasts.

##End of Project
#This notebook concludes the analytical workflow for the comparative flood risk assessment of Lagos and Durban. 
#Together, the seven notebooks provide a reproducible pipeline that progresses from data preparation through climate analysis, 
#urbanisation assessment, disaster analysis, statistical modelling, spatial mapping and economic evaluation
#to support evidence-based urban flood risk management.

